# Lab 3 — Structured tickets and the tool loop

*Day 2, hours 2 and 4 · 2 × 50 minutes · pairs*

::: {.callout-note appearance="simple"}
**Objective** — Part A: extract validated `ServiceTicket` objects from messy
bilingual citizen messages and measure the schema-pass rate. Part B: wire three
tools and the bounded loop, and pass the negative-test suite.

**Before you start** — Module 2's lab complete. `data/citizen_messages_50.jsonl` —
bilingual and deliberately messy: dialect, missing hamzas, Arabic-Indic digits,
mixed script, one 400-word polite ramble.

**You finish with** — six numbers in `BENCHMARKS.md`, an invented-field audit at
zero, and the tool-safety suite green.
:::

In [1]:
import os, pathlib, sys, re, subprocess, urllib.request, json

# pytest and ruff colour their output; those escapes render as noise once the
# notebook is published, so they come off here rather than per command.
ANSI = re.compile(chr(27) + "\[[0-9;]*m")

for cand in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
    if (cand / "src" / "murshid").is_dir():
        os.chdir(cand); break
    if (cand / "murshid" / "src" / "murshid").is_dir():
        os.chdir(cand / "murshid"); break

sys.path.insert(0, "src")
os.environ["PYTHONUTF8"] = "1"
os.environ.setdefault("PYTHONPATH", "src")

def run(*args, quiet_logs=True):
    """Run a course command and print what it printed.

    quiet_logs drops the structured log lines so the boxed summary is readable;
    pass quiet_logs=False when the log IS the lesson.
    """
    out = subprocess.run([sys.executable, *args], capture_output=True, text=True,
                         encoding="utf-8", errors="replace")
    text = ANSI.sub("", out.stdout + out.stderr)
    if quiet_logs:
        text = "\n".join(l for l in text.splitlines()
                          if not l.startswith("20") or "[" not in l[:40])
    print(text.strip())
    return out.returncode

# The gateway is 127.0.0.1 on a laptop and `gateway` inside compose, so take it
# from the same environment variable the application routes through rather than
# hardcoding a host that is only right in one of the two places.
GATEWAY = os.environ.get("MURSHID_PRIMARY_BASE_URL", "http://127.0.0.1:8080/v1")
GATEWAY = GATEWAY.rsplit("/v1", 1)[0].rstrip("/")

def fault(payload):
    """Fault injection on the course gateway: the 429 storm and the outage drill."""
    req = urllib.request.Request(
        GATEWAY + "/admin/fault", method="POST",
        data=json.dumps(payload).encode(), headers={"content-type": "application/json"})
    with urllib.request.urlopen(req, timeout=5) as r:
        return json.load(r)

def gateway_stats():
    with urllib.request.urlopen(GATEWAY + "/admin/stats", timeout=5) as r:
        return json.load(r)

try:
    with urllib.request.urlopen(GATEWAY + "/healthz", timeout=3) as r:
        print("gateway:", json.load(r)["models"])
except Exception:
    print(f"gateway at {GATEWAY} is NOT answering — start it first:")
    print("   make gateway      (or)   docker compose up -d gateway")
print("cwd:", pathlib.Path.cwd())

gateway: ['course-flagship', 'course-small', 'course-anthropic', 'murshid-onprem']
cwd: /srv


<>:5: SyntaxWarning: invalid escape sequence '\['
<>:5: SyntaxWarning: invalid escape sequence '\['
/tmp/ipykernel_107/1500538845.py:5: SyntaxWarning: invalid escape sequence '\['
  ANSI = re.compile(chr(27) + "\[[0-9;]*m")


# Part A — the ticket

## 1 · The contract (10 min)

The validators carry the rules no JSON Schema can express. Read both.

In [2]:
import inspect
from murshid.domain.ticket import Applicant
src = inspect.getsource(Applicant)
print(src[src.index("@field_validator"):][:1100])

@field_validator("national_id")
    @classmethod
    def valid_national_id(cls, v: str | None) -> str | None:
        if v is not None and not (len(v) == 10 and v.isdigit() and v[0] in "12"):
            raise ValueError("must be 10 digits starting with 1 (citizen) or 2 (resident)")
        return v

    @field_validator("phone")
    @classmethod
    def valid_phone(cls, v: str | None) -> str | None:
        if v is None:
            return v
        digits = v.replace(" ", "").replace("-", "")
        if not (digits.startswith(("+9665", "05", "9665")) and sum(c.isdigit() for c in digits) >= 9):
            raise ValueError("must be a Saudi mobile number, e.g. +9665XXXXXXXX")
        return digits



`if v is None: return v` — absent is legal, *invented* is not, and that distinction
is the never-invent rule expressed in a type. The phone validator normalises before
checking, so `05x xxx xxxx` and `+9665xxxxxxxx` are the same number to this
contract, and its error message names the expected shape because that message is
what the repair turn sees.

Confirm the contract still fits the strict-mode subset:

In [3]:
run("scripts/schema_check.py")

────────────────────────────────────────────────────────────────────────
schema-check | strict-mode subset
────────────────────────────────────────────────────────────────────────
  OK  service_ticket
  OK  applicant
  OK  booking_request
  OK  guard_verdict
  OK  route_verdict

5/5 contracts strict-safe


0

## 2 · The repair loop (15 min)

Extraction on a rich message, then on one with almost nothing in it.

In [4]:
from murshid.app import build_client
from murshid.config import get_settings
from murshid.pipeline.extract import extract_ticket

client = build_client(get_settings(), "primary")
ticket, outcome = extract_ticket(
    client, "السلام عليكم، اسمي فيصل العتيبي وأبغى أجدد السجل التجاري حقي في الرياض")
print("first try:", outcome.first_try, "| attempts:", outcome.attempts)
print(ticket.model_dump_json(indent=2)[:700])

{"schema": "service_ticket", "attempt": 1, "outcome": "first_try", "event": "structured_extracted", "level": "info", "timestamp": "2026-09-06T11:30:32.768081Z"}


first try: True | attempts: 1
{
  "service_type": "other",
  "summary_en": "Citizen asks about a government service in Riyadh.",
  "city": "Riyadh",
  "urgency": "routine",
  "language": "ar",
  "applicant": {
    "full_name": "فيصل العتيبي وأبغى أجدد",
    "national_id": null,
    "phone": null
  },
  "needs_human": false
}


In [5]:
ticket2, outcome2 = extract_ticket(client, "كيف أجدد رخصتي التجارية؟")
print("first try:", outcome2.first_try, "| attempts:", outcome2.attempts)
print("national_id:", ticket2.applicant.national_id)
print("phone      :", ticket2.applicant.phone)
print("city       :", ticket2.city)

{"schema": "service_ticket", "attempt": 1, "outcome": "first_try", "event": "structured_extracted", "level": "info", "timestamp": "2026-09-06T11:30:32.861643Z"}


first try: True | attempts: 1
national_id: None
phone      : None
city       : unknown


`None` is the correct answer. `extract_ticket.v3`'s never-invent rule plus its one
null example are what earn it. **An invented field is a defect; an empty one is a
fact.**

## 3 · Measure the corpus (15 min)

In [6]:
run("scripts/extract_corpus.py", "--audit")

────────────────────────────────────────────────────────────────────────
extract-corpus | route=primary+fallback
────────────────────────────────────────────────────────────────────────
50 messages | first-try pass: 45/50 (90%) | after repair: 48/50 (96%) | escalated: 2
   by language: ar 32/35 (91%) → 34/35 (97%) | en 12/14 (86%) → 13/14 (93%) | mixed 1/1 (100%) → 1/1 (100%)
   invented-field audit: 0 invented across 15 annotated cases
   escalated to human review: 2
     m010: [['urgency']]
     m013: [['urgency']]

   written: eval/out/extract_corpus_default.json
{"schema": "service_ticket", "attempt": 1, "outcome": "first_try", "event": "structured_extracted", "level": "info", "timestamp": "2026-09-06T11:30:34.491945Z"}
{"schema": "service_ticket", "attempt": 1, "outcome": "first_try", "event": "structured_extracted", "level": "info", "timestamp": "2026-09-06T11:30:34.578860Z"}
{"schema": "service_ticket", "attempt": 1, "outcome": "first_try", "event": "structured_extracted", "leve

0

All six numbers go in `BENCHMARKS.md`. The two escalations are not a bug: one
repair, then a designed hand-off. **A corpus where nothing ever escalates is not
testing the failure path.**

## 4 · The comparison (10 min)

In [7]:
run("scripts/extract_corpus.py", "--route", "vllm", "--audit")

────────────────────────────────────────────────────────────────────────
extract-corpus | route=vllm
────────────────────────────────────────────────────────────────────────
50 messages | first-try pass: 42/50 (84%) | after repair: 49/50 (98%) | escalated: 1
   by language: ar 28/35 (80%) → 34/35 (97%) | en 13/14 (93%) → 14/14 (100%) | mixed 1/1 (100%) → 1/1 (100%)
   invented-field audit: 0 invented across 15 annotated cases
   escalated to human review: 1
     m031: [['urgency']]

   written: eval/out/extract_corpus_vllm.json
{"schema": "service_ticket", "attempt": 1, "outcome": "first_try", "event": "structured_extracted", "level": "info", "timestamp": "2026-09-06T11:30:41.567659Z"}
{"schema": "service_ticket", "attempt": 1, "outcome": "first_try", "event": "structured_extracted", "level": "info", "timestamp": "2026-09-06T11:30:41.672844Z"}
{"schema": "service_ticket", "attempt": 1, "outcome": "first_try", "event": "structured_extracted", "level": "info", "timestamp": "2026-09-06T11

0

::: {.callout-warning}
## Then write a sentence about error bars

Fifty cases carry roughly ±6 points of noise, so a few points between routes after
repair is a coin, not a finding. The differences that *are* real are the first-try
rates and the size of the gap the repair loop closes. Learning which differences
survive their error bars is most of what Module 5 is about.
:::

# Part B — the tool loop

## 5 · Tool descriptions route (10 min)

The smoke suite asserts *which* tools fire, including the cases where none should.

In [8]:
run("scripts/tool_smoke.py")

────────────────────────────────────────────────────────────────────────
tool-smoke
────────────────────────────────────────────────────────────────────────
  OK  status lookup with a reference                   called=['check_application_status'] expected=['check_application_status']
  OK  documents question — must NOT call a tool        called=[] expected=[]
  OK  status question without a reference — must ask, not guess called=[] expected=[]
  OK  booking with everything confirmed                called=['book_appointment'] expected=['book_appointment']
  OK  asks for a human                                 called=['escalate_to_agent'] expected=['escalate_to_agent']

5/5 as expected
{"route": "primary+fallback", "faq_alias": "murshid-default", "service_alias": "murshid-default", "routing_enabled": false, "cascade": false, "cache": false, "semantic_cache": false, "faq_prompt": "answer_faq.v5", "event": "assistant_built", "level": "info", "timestamp": "2026-09-06T11:30:49.322119Z"}
{"s

0

Now break it on purpose. Descriptions route — one over-broad sentence and the tool
fires on everything.

The smoke script has to run **in this kernel** for the edit to take effect, so
import its `main` rather than shelling out: a subprocess would load its own copy of
the registry and the change would vanish.

In [9]:
import importlib, sys
sys.path.insert(0, "scripts")
tool_smoke = importlib.import_module("tool_smoke")

from murshid.tools import registry
tool = registry.BY_NAME["check_application_status"]
original = tool.description
print("before:", original[:100], "...")

before: Look up the current status of a government application by its reference number (format: two letters  ...


In [10]:
tool.description = "Use for any question about applications."
tool_smoke.main()

{"route": "primary+fallback", "faq_alias": "murshid-default", "service_alias": "murshid-default", "routing_enabled": false, "cascade": false, "cache": false, "semantic_cache": false, "faq_prompt": "answer_faq.v5", "event": "assistant_built", "level": "info", "timestamp": "2026-09-06T11:30:50.801812Z"}


{"schema": "guard_verdict", "attempt": 1, "outcome": "first_try", "event": "structured_extracted", "trace_id": "08c5642b3117", "level": "info", "timestamp": "2026-09-06T11:30:50.845422Z"}


{"route": "cheap", "intent": "guard", "stage": "input_guard", "model_id": "course-small", "prompt_version": "input_guard_classifier.v2", "input_tokens": 246, "cached_tokens": 0, "output_tokens": 6, "latency_ms": 32.6, "cost_halalas": 0.015126, "trace_id": "08c5642b3117", "cache_tier": "", "event": "llm_cost", "level": "info", "timestamp": "2026-09-06T11:30:50.846472Z"}


{"schema": "route_verdict", "attempt": 1, "outcome": "first_try", "event": "structured_extracted", "level": "info", "timestamp": "2026-09-06T11:30:50.882187Z"}


{"route": "cheap", "intent": "router", "stage": "router", "model_id": "course-small", "prompt_version": "route_intent.v1", "input_tokens": 205, "cached_tokens": 0, "output_tokens": 6, "latency_ms": 31.4, "cost_halalas": 0.01283, "trace_id": "", "cache_tier": "", "event": "llm_cost", "level": "info", "timestamp": "2026-09-06T11:30:50.883143Z"}


...

{"route": "cheap", "intent": "router", "stage": "router", "model_id": "course-small", "prompt_version": "route_intent.v1", "input_tokens": 205, "cached_tokens": 0, "output_tokens": 8, "latency_ms": 31.5, "cost_halalas": 0.01328, "trace_id": "", "cache_tier": "", "event": "llm_cost", "level": "info", "timestamp": "2026-09-06T11:30:51.735415Z"}


{"intent": "escalate", "prompt_version": "route_intent.v1", "event": "routed", "level": "info", "timestamp": "2026-09-06T11:30:51.736103Z"}


{"reason": "router sent this conversation to a human", "event": "escalated_to_agent", "level": "info", "timestamp": "2026-09-06T11:30:51.737974Z"}


  BAD booking with everything confirmed                called=['check_application_status'] expected=['book_appointment']
      reply: I couldn't complete that. Ask the citizen to confirm the reference number: two letters and eight dig
  OK  asks for a human                                 called=['escalate_to_agent'] expected=['escalate_to_agent']

3/5 as expected


1

One over-broad sentence and the documents question now fires the status tool.
**Descriptions route** — which is why the `don't` cases in a description are not
padding.

Put it back before moving on; the rest of the lab depends on it.

In [11]:
tool.description = original
tool_smoke.main()

{"route": "primary+fallback", "faq_alias": "murshid-default", "service_alias": "murshid-default", "routing_enabled": false, "cascade": false, "cache": false, "semantic_cache": false, "faq_prompt": "answer_faq.v5", "event": "assistant_built", "level": "info", "timestamp": "2026-09-06T11:30:51.765038Z"}


{"schema": "guard_verdict", "attempt": 1, "outcome": "first_try", "event": "structured_extracted", "trace_id": "f60ac55df2ad", "level": "info", "timestamp": "2026-09-06T11:30:51.800438Z"}


{"route": "cheap", "intent": "guard", "stage": "input_guard", "model_id": "course-small", "prompt_version": "input_guard_classifier.v2", "input_tokens": 246, "cached_tokens": 0, "output_tokens": 6, "latency_ms": 33.3, "cost_halalas": 0.015126, "trace_id": "f60ac55df2ad", "cache_tier": "", "event": "llm_cost", "level": "info", "timestamp": "2026-09-06T11:30:51.801258Z"}


{"schema": "route_verdict", "attempt": 1, "outcome": "first_try", "event": "structured_extracted", "level": "info", "timestamp": "2026-09-06T11:30:51.833884Z"}


{"route": "cheap", "intent": "router", "stage": "router", "model_id": "course-small", "prompt_version": "route_intent.v1", "input_tokens": 205, "cached_tokens": 0, "output_tokens": 6, "latency_ms": 30.6, "cost_halalas": 0.01283, "trace_id": "", "cache_tier": "", "event": "llm_cost", "level": "info", "timestamp": "2026-09-06T11:30:51.834715Z"}


...

{"route": "cheap", "intent": "router", "stage": "router", "model_id": "course-small", "prompt_version": "route_intent.v1", "input_tokens": 205, "cached_tokens": 0, "output_tokens": 8, "latency_ms": 30.9, "cost_halalas": 0.01328, "trace_id": "", "cache_tier": "", "event": "llm_cost", "level": "info", "timestamp": "2026-09-06T11:30:52.584435Z"}


{"intent": "escalate", "prompt_version": "route_intent.v1", "event": "routed", "level": "info", "timestamp": "2026-09-06T11:30:52.585206Z"}


{"reason": "router sent this conversation to a human", "event": "escalated_to_agent", "level": "info", "timestamp": "2026-09-06T11:30:52.586849Z"}


  OK  booking with everything confirmed                called=['book_appointment'] expected=['book_appointment']
  OK  asks for a human                                 called=['escalate_to_agent'] expected=['escalate_to_agent']

5/5 as expected


0

## 6 · Bounds and the authorisation gate (15 min)

Every negative test exercises `_execute`, where the *order* of the checks is the
security model.

In [12]:
run("-m", "pytest", "tests/pipeline/test_tool_safety.py", "-v", "--no-header", "-q")

..........                                                               [100%]
10 passed in 0.05s


0

::: {.callout-important}
## Where else could that identity check live?

List the alternatives before reading on: in the prompt; in the tool description; in
the model's good judgement.

Now ask what happens to each under Module 4's injection scenarios. **Every answer
that lives inside the token stream is an answer an attacker can write to.** The
session object is the only one that is not.
:::

Watch the gate refuse a cross-citizen booking directly.

In [13]:
from murshid.domain.session import Session

s = Session(citizen_id="1012345678", identity_verified=True)
for label, args in [("own account ", {"citizen_id": "1012345678"}),
                    ("someone else", {"citizen_id": "2098765432"})]:
    v = s.authorize("book_appointment", args)
    print(f"{label}: allowed={v.allowed} reason={v.reason}")
    if v.user_hint:
        print(f"              hint: {v.user_hint}")

{"session": "sess_e323478a1c", "tool": "book_appointment", "requested_for": "2098765432", "event": "authz_cross_citizen_denied", "level": "warning", "timestamp": "2026-09-06T11:30:54.273613Z"}


own account : allowed=True reason=
someone else: allowed=False reason=cross_citizen
              hint: I can only act on your own account. Each person books their own appointment from their own account.


The argument is *read*, but `self.citizen_id` is what it is compared against, and
the verdict carries a `user_hint` because a refusal a citizen cannot act on is a
dead end rather than a guardrail.

## 7 · End to end (15 min)

A booking, and then the audit trail it left.

In [14]:
run("-m", "murshid.cli", "ask", "أريد حجز موعد في الأحوال المدنية بالرياض بتاريخ 2026-10-14، أكّد الحجز")

[service → course-flagship via primary] 639ms, 3435 in (3435 cached) / 67 out, 0.765 halalas
تم الحجز. رقم التأكيد APF49DD29B في Riyadh بتاريخ 2026-10-14.
{"route": "primary+fallback", "faq_alias": "murshid-default", "service_alias": "murshid-default", "routing_enabled": false, "cascade": false, "cache": false, "semantic_cache": false, "faq_prompt": "answer_faq.v5", "event": "assistant_built", "level": "info", "timestamp": "2026-09-06T11:30:55.671344Z"}
{"schema": "guard_verdict", "attempt": 1, "outcome": "first_try", "event": "structured_extracted", "trace_id": "5b73b47033a9", "level": "info", "timestamp": "2026-09-06T11:30:56.117783Z"}
{"route": "cheap", "intent": "guard", "stage": "input_guard", "model_id": "course-small", "prompt_version": "input_guard_classifier.v2", "input_tokens": 258, "cached_tokens": 0, "output_tokens": 6, "latency_ms": 417.2, "cost_halalas": 0.015798, "trace_id": "5b73b47033a9", "cache_tier": "", "event": "llm_cost", "level": "info", "timestamp": "2026-09-06T

0

## 8 · Commit (10 min)

```bash
git commit -am "feat: validated ticket extraction and the bounded tool loop"
```

## If you finish early

Add parallel execution for read-only calls, and prove side-effecting calls still
serialise. Then argue about the fourth risk class: lab results are read-only
*technically* but sensitive. Does the registry need another class, or does
authorisation already cover it?